# Music Review Insight Finder
## Collection and preparation: readable historical copy

Objective Question: Can an adapted large-language model user comments into categories (specific musical observations,
general opinions, and background information) from online music reviews?

This is a clearer copy of the original October 15 workflow. Only the project-folder setup was changed to a relative repository path for this
submission copy. All remaining executable logic, settings, cell order, execution
counts, and saved outputs are unchanged. It has
not been rerun during this cleanup. The original contains no saved code outputs.

Sections 5–8 show the earlier manual-label workflow. The later AI-audited splits
used in experiments 06–08 are a separate workflow; running this notebook does not
recreate those experiments. Use `10_saved_results_review_readable.ipynb` to review
their actual results and `09_vader_sentiment_readable.ipynb` for sentiment EDA.

Use the existing tf_env Python environment. Running cells can collect uncached
reviews and write to the original session folder; this copy does not redirect
those writes. Read the historical sections before choosing to run them.

## 1. Project set up

In [ ]:
from pathlib import Path
import sys, json, hashlib
from collections import Counter

# Set the project folder.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "music_review_pipeline.py").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
assert (PROJECT_ROOT / "music_review_pipeline.py").exists(), "Keep the supplied .py files beside this notebook."
sys.path.insert(0, str(PROJECT_ROOT))
import music_review_pipeline as pipeline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

SEED = 509
SESSION_NAME = "2026-10-15_music_insight_v1"
RUN_DIR = PROJECT_ROOT / "music_review_runs" / SESSION_NAME
RUN_DIR.mkdir(parents=True, exist_ok=True)
CHART_DIR = RUN_DIR / "charts"
CHART_DIR.mkdir(exist_ok=True)
print("Python:", sys.version.split()[0])
print("Project:", PROJECT_ROOT)
print("Session:", RUN_DIR)
print("Libraries:", {"pandas": pd.__version__, "numpy": np.__version__})

The session folder holds cached pages, prepared data, and annotation edits.
Use a new session name if collection settings, cleaning rules, or labels change.
This submission copy resolves the project folder relative to the repository; other computations are unchanged.

## 2. Collect reviews

The collector requests English text reviews from CritiqueBrainz (up to 50 pages). Cached pages are reused. Uncached pages require public API requests.

A release group can be an album, EP, or single. The bounded, oldest-first sample
does not represent all music listeners. Source details are retained.

In [ ]:
# Collect up to 500 reviews.
reviews = pipeline.collect_reviews(RUN_DIR, max_reviews=500, max_pages=30, page_size=50)
print("Saved reviews:", len(reviews))
print("Data file:", RUN_DIR / "reviews.csv")
display(pd.DataFrame(reviews)[["review_id", "entity_type", "language", "license_id", "author"]].head())

## 3. Prepare passages and check review lengths

In [ ]:
# Read the saved reviews.
reviews = pipeline.load_csv(RUN_DIR / "reviews.csv")
word_counts = [len(review["text_raw"].split()) for review in reviews]

def avg_word_count(counts):
    return sum(counts) / len(counts) if counts else None

min_word_count = min(word_counts) if word_counts else None
max_word_count = max(word_counts) if word_counts else None
summary = {"reviews": len(word_counts), "min_words": min_word_count,
           "max_words": max_word_count, "average_words": avg_word_count(word_counts)}
print(summary)
assert avg_word_count([]) is None and avg_word_count([0]) == 0.0
pipeline.save_json(RUN_DIR / "word_count_summary.json", summary)
print("Empty-input checks passed; no stray 'and c' condition is used.")

In [ ]:
passages, cleaning_audit = pipeline.make_passages(reviews, max_words=100, min_words=3)
if not passages:
    raise ValueError("No usable passages. Inspect raw reviews and cleaning settings.")
pipeline.save_json(RUN_DIR / "passages.json", passages)
pipeline.save_csv(RUN_DIR / "passages.csv", passages, pipeline.PASSAGE_FIELDS)
pipeline.save_json(RUN_DIR / "cleaning_audit.json", cleaning_audit)
print("Unique passages:", len(passages), "Cleaning exclusions:", cleaning_audit)
print("Reviews missing license identifiers:", sum(not r["license_id"] for r in reviews))
print("Reviews missing author attribution:", sum(not r["author"] for r in reviews))
display(pd.DataFrame(passages)[["passage_id", "text", "word_count"]].head(8))

Raw review text stays in `reviews.csv`. Preparation removes HTML and extra
whitespace, keeps case and punctuation, and splits long text into passages of at
most 100 words. Very short passages and exact duplicates are counted and removed.
Splitting can sometimes lose context. Ratings are not used as task labels.

Author, source, and license fields remain with the records. Review those terms
before redistributing text or adapters.

## 4. Explore lengths and frequent words

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

frame = pd.DataFrame(passages)
lengths = frame["word_count"].astype(int)
vectorizer = CountVectorizer(stop_words="english", max_features=2000)
try:
    matrix = vectorizer.fit_transform(frame["text"])
    frequencies = pd.Series(np.asarray(matrix.sum(axis=0)).ravel(), index=vectorizer.get_feature_names_out())
    top_terms = frequencies.sort_values(ascending=False).head(15)
except ValueError:  # Handle passages with no usable terms.
    top_terms = pd.Series(dtype=float)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].hist(lengths, bins=min(20, max(1, lengths.nunique())), color="#168477", edgecolor="white")
axes[0].set(title="Passage lengths", xlabel="Whitespace-separated words", ylabel="Passages")
if not top_terms.empty:
    top_terms.sort_values().plot.barh(ax=axes[1], color="#168477")
axes[1].set(title="Frequent terms after English stopword removal", xlabel="Occurrences")
fig.tight_layout()
chart_path = CHART_DIR / "01_lengths_and_frequencies.png"
if not chart_path.exists():
    fig.savefig(chart_path, dpi=160, bbox_inches="tight")
plt.show()

license_counts = Counter(r["license_id"] or "MISSING" for r in reviews)
entity_counts = Counter(r["entity_id"] or "MISSING" for r in reviews)
display(pd.DataFrame(license_counts.items(), columns=["license", "reviews"]))
display(pd.DataFrame(entity_counts.most_common(10), columns=["release_group_id", "reviews"]))
print("Text structure summary:")
display(lengths.describe().to_frame("word_count"))
print("Term counts are descriptive, not inferred topics or model performance.")

## 5. Prepare manual-labelling file

In [ ]:
reference_path = RUN_DIR / "annotation_reference.json"
annotation_path = RUN_DIR / "annotation.csv"
if reference_path.exists():
    reference = json.loads(reference_path.read_text(encoding="utf-8"))
else:
    reference = pipeline.annotation_sample(passages, limit=300, per_review=3, seed=SEED)
    pipeline.save_json(reference_path, reference)
if not annotation_path.exists():
    pipeline.save_csv(annotation_path, reference, pipeline.ANNOTATION_FIELDS)
else:
    print("Preserved existing annotation.csv, including your edits.")
print("Label this file:", annotation_path)
print("Instructions:", PROJECT_ROOT / "LABELING_GUIDE.md")
print("Rows:", len(reference), "Allowed labels:", pipeline.LABELS)
display(pd.DataFrame(pipeline.load_csv(annotation_path))[["passage_id", "text", "label", "status"]].head())

This historical section creates the manual annotation CSV. Follow the
labeling guide, keep IDs and source text unchanged, and record exclusions.

Later experiments used AI-audited labels across all splits. Those labels are
AI-assisted references, not independent human gold. This earlier CSV workflow
must not be described as the provenance of the final audited evaluation.

## 6. Check the earlier labels and freeze groups

In [ ]:
labeled_rows, annotation_audit = pipeline.load_annotations(annotation_path, reference)
pipeline.save_json(RUN_DIR / "annotation_audit.json", annotation_audit)
splits = pipeline.freeze_splits(RUN_DIR, labeled_rows, seed=SEED)
print(annotation_audit)
split_counts = pd.DataFrame({name: Counter(r["label"] for r in rows) for name, rows in splits.items()}).fillna(0).astype(int)
display(split_counts)
ax = split_counts.plot.bar(figsize=(11, 4), color=["#168477", "#54a6a6", "#b0d5c8"])
ax.set(title="Human-labeled category distributions", ylabel="Passages", xlabel="Label")
plt.xticks(rotation=12, ha="right")
plt.tight_layout()
if not (CHART_DIR / "02_labels_by_split.png").exists():
    plt.savefig(CHART_DIR / "02_labels_by_split.png", dpi=160, bbox_inches="tight")
plt.show()
print("Frozen splits:", RUN_DIR / "splits")

The earlier split targets are about 60% training, 20% validation, and 20%
test, grouped by release ID (review ID fallback). They are proposed settings,
not course requirements.

The unchanged chart title below says “Human-labeled.” That title belongs to this
historical manual workflow; it does not describe the final AI-audited labels.
The original test set was later viewed in run 06. Follow-ups A–D used validation
only, and their results are exploratory. Do not rerun the old test as a new final
confirmation.

## 7. Review the earlier simple baselines

In [ ]:
majority = pipeline.majority_label(splits["train"])
validation = splits["validation"]
baseline_predictions = {
    "majority": [majority] * len(validation),
    "keyword": [pipeline.keyword_prediction(r["text"]) for r in validation],
}
validation_results = {name: pipeline.evaluate_predictions(validation, preds)
                      for name, preds in baseline_predictions.items()}
pipeline.save_json(RUN_DIR / "validation_simple_baselines.json", validation_results)
display(pd.DataFrame({name: {k: result[k] for k in ("n", "accuracy", "macro_f1")}
                      for name, result in validation_results.items()}).T)
print("Keyword rules are deliberately simple. Examine validation errors; do not tune them on test data.")

## 8. Export the earlier training bundle

In [ ]:
bundle_path = pipeline.make_training_bundle(RUN_DIR, PROJECT_ROOT)
print("Local training archive:", bundle_path)
print("Next notebook:", PROJECT_ROOT / "03_music_review_lora_COLAB.ipynb")
print("No dataset has been uploaded. Choose yourself whether to transfer this archive to Colab.")

The unchanged export code points to notebook 03 and the earlier split
workflow. The final executed evidence is in the preserved notebooks 06, 07, and 08 in `training/`;
see the source map in `README.md`.

For the presentation, explain collection, cleaning, labels, model comparisons,
errors, and limitations. The course requires a documented GitHub code URL and a
10–15 minute recorded presentation, including a 2–3 minute GenAI reflection.